# Serve Qwen2.5-7B-Instruct on a free Colab GPU

This notebook loads an open-source LLM (Qwen2.5-7B-Instruct, 4-bit quantized) and exposes it as an OpenAI-compatible `/v1/chat/completions` endpoint via a public ngrok tunnel. Your local/hosted gateway calls this URL as its `COLAB_ENDPOINT`.

**Before running:**
1. Runtime -> Change runtime type -> T4 GPU (free tier).
2. Get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken and paste it in the cell below.
3. Run all cells top to bottom. Loading the model takes a few minutes.

**Model choice:** Qwen2.5-7B-Instruct fits comfortably in 4-bit on a T4's 16GB VRAM. Swap `MODEL_ID` below for `meta-llama/Llama-3.1-8B-Instruct` if you prefer Llama (you'll need to accept its license on Hugging Face first and pass an HF token).

In [ ]:
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest-asyncio

In [ ]:
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # or "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = ""  # only needed for gated models like Llama; leave blank for Qwen

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
model_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **tok_kwargs)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    **model_kwargs,
)
print("Model loaded.")

In [ ]:
import time
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: list[Message]
    max_tokens: int = 512
    temperature: float = 0.7

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    chat = [{"role": m.role, "content": m.content} for m in req.messages]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            temperature=req.temperature,
            do_sample=req.temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "id": f"colab-{int(time.time()*1000)}",
        "object": "chat.completion",
        "model": MODEL_ID,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": text},
            "finish_reason": "stop",
        }],
    }

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTHTOKEN)

public_url = ngrok.connect(8000)
print(f"\n=== Public endpoint: {public_url} ===")
print("Copy this into your gateway's .env as COLAB_ENDPOINT (no trailing slash).\n")

uvicorn.run(app, host="0.0.0.0", port=8000)

## Notes
- **Session persistence:** Colab free tier disconnects after ~90 min idle or a few hours of use. This is fine for building/testing/demoing but not for a real production deployment -- treat the ngrok URL as ephemeral and re-copy it into your gateway `.env` whenever you restart.
- **Quick test** from another cell or a terminal:
```bash
curl -X POST <public_url>/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"messages": [{"role": "user", "content": "Hello, who are you?"}]}'
```
- Keep this tab open/active while you're running eval or red-team suites against it -- Colab will kill the runtime if it thinks the tab is idle.